In [11]:
import os
from utils import extract_body, tokenize, clean_tokens, decode
from utils import chunk_tokens, flatten_token_chunks
from utils import extract_few_shot_examples
from utils import select_few_shot 
from utils import merge_tokens_with_auto_labels, add_style_and_parent_to_auto_labels, compare_html_allow_auto_labels
from models import GPTAssistant
from process_chunks import process_chunks

In [12]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 10  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [13]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2019SCC65"
round = "ronde_1"
anno = "llm"
version = "v1"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.htm"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"

os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_1\plain_html_arbre_balise\2019SCC65.htm
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [14]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


   ⚠ Warning: stop_bookmark_separation=True but bookmark not found
   ✓ Chunked tokens into 285 chunks (>= 500 tokens each)


In [15]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 23066
   ✓ Splitting: 23066 tokens before, 31001 tokens after
   ✓ Chunked tokens into 36 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Total chunks: 36 before + 62 after = 98


In [16]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [17]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 36 few-shot examples from chunks
   ✓ Selected 10 few-shot examples for processing.


In [19]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [ ]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils\prompts\simplified_parent_extraction_cot.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 285 chunks with LLM...
   ✓ Using 10 few-shot examples


Processing chunks:   0%|          | 0/285 [00:00<?, ?it/s]

## Post Processing

In [15]:
# ---------- Merge all tokens ----------
processed_tokens_flat = flatten_token_chunks(processed_chunks)


original_tokens = tokenize(html_content)
processed_html_content = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

processed_html = decode(processed_html_content)

print(f"\nMerged HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = decode(add_style_and_parent_to_auto_labels(processed_html))


# ---------- Compare with original HTML (ignoring auto_label tags) ----------
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)



   ✓ Flattened 179 chunks into 90476 tokens

Merged HTML length: 283753
   ✓ HTMLs match when ignoring auto_label tags


In [16]:
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{anno}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA
